# Prepare DepMap PRISM 24Q2 snapshot for HakaseAI L3 Stage 4

**Purpose** — preprocess **DepMap PRISM 24Q2** (per-cell-line CRISPR Chronos essentiality + CCLE expression) into a small, query-friendly Parquet that the Replit-hosted `ai-service` loads at runtime to power the **L3 Stage 4 patient-line projection** endpoint (`/predict/patient-line`).

**Why off-platform?** The raw Chronos + expression matrices are several GB and joining them per (cell_line × gene) needs ~16 GB RAM. Replit dev containers can't comfortably do that. We do the heavy lift here in Colab (high-RAM runtime), emit a ~50–150 MB Parquet, then upload that Parquet to `artifacts/ai-service/cache/depmap_prism_aggregated.parquet` in the Replit project.

**Output schema (every column required — the loader rejects Parquets missing any of these):**

| column | type | meaning |
| --- | --- | --- |
| `depmap_id` | str (uppercase) | `ACH-NNNNNN` |
| `cell_line_name` | str | stripped name (CCLE_Name without lineage suffix) |
| `primary_disease` | str | e.g. `Acute Myeloid Leukemia` |
| `lineage` | str | e.g. `myeloid` (lowercase) |
| `target_gene_symbol` | str (uppercase) | HUGO symbol |
| `target_essentiality_chronos` | float32 | DepMap CRISPR Chronos gene effect (<0 essential, ~0 neutral, >0 anti) |
| `target_expression_log2_tpm` | float32 | CCLE expression in `log2(TPM + 1)` |

**Provenance** — attach `snapshot_name`, `snapshot_release`, `preparation_date`, `n_cell_lines`, `n_target_genes`, `n_oncology_lineages`, `notebook_version` to the Parquet's pyarrow metadata so the running service can echo them back honestly.

**Honesty contract** — we do NOT fabricate rows. If a (cell_line × gene) pair is missing in the source files we drop it; we do NOT impute. The downstream endpoint reports `out-of-domain` for any query gene absent from this Parquet.

## 1 · Inputs (DepMap 24Q2 release)

Download from <https://depmap.org/portal/data_page/?tab=allData>:
- `Model.csv`                — cell-line metadata (`ModelID`, `StrippedCellLineName`, `OncotreePrimaryDisease`, `OncotreeLineage`)
- `CRISPRGeneEffect.csv`     — Chronos gene-effect matrix (cell_lines × genes)
- `OmicsExpressionProteinCodingGenesTPMLogp1.csv` — CCLE log2(TPM+1) expression

Pin the release tag (e.g. `24Q2`) so the snapshot is reproducible.

In [ ]:
RELEASE = '24Q2'
NOTEBOOK_VERSION = 'v1.0'

# Path setup — adjust to wherever you stash the DepMap files in Colab.
MODEL_CSV   = '/content/depmap/Model.csv'
CRISPR_CSV  = '/content/depmap/CRISPRGeneEffect.csv'
EXPR_CSV    = '/content/depmap/OmicsExpressionProteinCodingGenesTPMLogp1.csv'

# Restrict to oncology lineages we want to project against. The endpoint refuses
# to run for non-oncology compounds anyway, so non-oncology models are dropped.
ONCOLOGY_LINEAGES_EXCLUDE = {'normal', 'fibroblast', 'non_cancerous'}

In [ ]:
import pandas as pd
import numpy as np
import re

model = pd.read_csv(MODEL_CSV)
model = model.rename(columns={
    'ModelID': 'depmap_id',
    'StrippedCellLineName': 'cell_line_name',
    'OncotreePrimaryDisease': 'primary_disease',
    'OncotreeLineage': 'lineage',
})[['depmap_id', 'cell_line_name', 'primary_disease', 'lineage']]
model['lineage'] = model['lineage'].astype(str).str.lower().str.strip()
model = model[~model['lineage'].isin(ONCOLOGY_LINEAGES_EXCLUDE)]
print(f'Oncology cell lines: {len(model)}')

In [ ]:
# DepMap CSVs are wide: rows = cell lines, columns = 'GENE_SYMBOL (Entrez)'.
# Melt to long, parse out the HUGO symbol.
_GENE_RE = re.compile(r'^([A-Za-z0-9\-]+)\s*\(\d+\)$')

def melt_depmap_wide(csv_path: str, value_name: str) -> pd.DataFrame:
    wide = pd.read_csv(csv_path, low_memory=False)
    wide = wide.rename(columns={wide.columns[0]: 'depmap_id'})
    long = wide.melt(id_vars=['depmap_id'], var_name='gene_label', value_name=value_name)
    sym = long['gene_label'].str.extract(_GENE_RE)[0]
    long['target_gene_symbol'] = sym.astype(str).str.upper()
    long = long.dropna(subset=['target_gene_symbol', value_name])
    long = long[['depmap_id', 'target_gene_symbol', value_name]]
    return long

essentiality = melt_depmap_wide(CRISPR_CSV, 'target_essentiality_chronos')
expression   = melt_depmap_wide(EXPR_CSV,   'target_expression_log2_tpm')
print(essentiality.shape, expression.shape)

In [ ]:
# Inner-join on (depmap_id, gene) — drop pairs that aren't in BOTH matrices.
joined = essentiality.merge(expression, on=['depmap_id', 'target_gene_symbol'], how='inner')
joined = joined.merge(model, on='depmap_id', how='inner')
joined['depmap_id'] = joined['depmap_id'].astype(str).str.upper()
joined['target_gene_symbol'] = joined['target_gene_symbol'].astype(str).str.upper()
joined['target_essentiality_chronos'] = joined['target_essentiality_chronos'].astype('float32')
joined['target_expression_log2_tpm']  = joined['target_expression_log2_tpm'].astype('float32')
joined = joined[[
    'depmap_id', 'cell_line_name', 'primary_disease', 'lineage',
    'target_gene_symbol', 'target_essentiality_chronos', 'target_expression_log2_tpm',
]]
joined = joined.dropna()
print(f'Final rows: {len(joined):,}')
print(f'Cell lines: {joined.depmap_id.nunique():,}  Genes: {joined.target_gene_symbol.nunique():,}  Lineages: {joined.lineage.nunique()}')

In [ ]:
# Write Parquet with provenance metadata embedded so the running service can echo it.
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime, timezone

OUT_PATH = '/content/depmap_prism_aggregated.parquet'

tbl = pa.Table.from_pandas(joined, preserve_index=False)
meta = {
    'snapshot_name':        'DepMap PRISM',
    'snapshot_release':     RELEASE,
    'preparation_date':     datetime.now(timezone.utc).isoformat(),
    'n_cell_lines':         str(joined.depmap_id.nunique()),
    'n_target_genes':       str(joined.target_gene_symbol.nunique()),
    'n_oncology_lineages':  str(joined.lineage.nunique()),
    'notebook_version':     NOTEBOOK_VERSION,
}
schema_meta = {**(tbl.schema.metadata or {}), **{k.encode(): v.encode() for k, v in meta.items()}}
tbl = tbl.replace_schema_metadata(schema_meta)
pq.write_table(tbl, OUT_PATH, compression='zstd')
print(f'Wrote {OUT_PATH}')

## 4 · Upload

Download `depmap_prism_aggregated.parquet` from Colab and place it at:

```
artifacts/ai-service/cache/depmap_prism_aggregated.parquet
```

Then restart the **AI Service (Python)** workflow. The `/predict/patient-line` endpoint will lazy-load the snapshot on the first request and echo the embedded provenance back in `model_info.snapshot`.